# iCare Data Processing
Load iCare CSV files from Lakehouse Files, clean and transform, then save as Delta tables.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

BASE_PATH = "Files/icare/icare_export"

## 1. Assets (dimension table)

In [ ]:
df_assets = (spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{BASE_PATH}/Assets.csv")
)

df_assets.write.mode("overwrite").format("delta").saveAsTable("icare_assets")
print(f"icare_assets: {df_assets.count()} rows")
df_assets.show(10, truncate=False)

## 2. Analyses

In [ ]:
schema_analyses = StructType([
    StructField("_id", StringType()),
    StructField("deadline", StringType()),
    StructField("asset_id", StringType()),
    StructField("analysis", StringType()),
    StructField("current_status", StringType()),
    StructField("technology", StringType()),
    StructField("created", StringType()),
    StructField("date", StringType()),
    StructField("measured_date", StringType()),
    StructField("planned_date", StringType()),
    StructField("updated", StringType()),
])

df_analyses = (spark.read.option("header", True).schema(schema_analyses)
    .option("multiLine", True).option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(f"{BASE_PATH}/Analyses.csv")
    .withColumn("deadline", F.when(F.col("deadline").startswith("0001"), None).otherwise(F.to_timestamp("deadline")))
    .withColumn("created", F.when(F.col("created").startswith("0001"), None).otherwise(F.to_timestamp("created")))
    .withColumn("date", F.when(F.col("date").startswith("0001"), None).otherwise(F.to_timestamp("date")))
    .withColumn("measured_date", F.when(F.col("measured_date").startswith("0001"), None).otherwise(F.to_timestamp("measured_date")))
    .withColumn("planned_date", F.when(F.col("planned_date").startswith("0001"), None).otherwise(F.to_timestamp("planned_date")))
    .withColumn("updated", F.when(F.col("updated").startswith("0001"), None).otherwise(F.to_timestamp("updated")))
)

df_analyses.write.mode("overwrite").format("delta").saveAsTable("icare_analyses")
print(f"icare_analyses: {df_analyses.count()} rows")
df_analyses.show(5, truncate=40)

## 3. DataPoints (fact table — chunked CSVs)

In [ ]:
df_datapoints = (spark.read.option("header", True).option("inferSchema", True)
    .option("multiLine", True).option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(f"{BASE_PATH}/DataPoints_*.csv")
    .withColumn("acqend", F.to_timestamp("acqend"))
    .withColumn("value", F.col("value").cast("double"))
    .withColumn("status", F.col("status").cast("int"))
)

df_datapoints.write.mode("overwrite").format("delta").saveAsTable("icare_datapoints")
print(f"icare_datapoints: {df_datapoints.count()} rows")
df_datapoints.select("measurementId", "res_type", "global_type", "acqend", "AssetId", "value").show(10, truncate=60)

## 4. Enriched DataPoints — join with Assets and Analyses

In [ ]:
df_enriched = (spark.table("icare_datapoints").alias("dp")
    .join(spark.table("icare_assets").alias("a"),
          F.col("dp.AssetId") == F.col("a._id"), "left")
    .join(spark.table("icare_analyses").alias("an"),
          F.col("dp.AssetId") == F.col("an.asset_id"), "left")
    .select(
        F.col("dp.measurementId"),
        F.col("dp.res_type"),
        F.col("dp.global_type"),
        F.col("dp.acqend"),
        F.col("dp.value"),
        F.col("dp.status"),
        F.col("dp.AssetId"),
        F.col("a.name").alias("asset_name"),
        F.col("a.gds"),
        F.col("an.current_status").alias("analysis_status"),
        F.col("an.technology"),
        F.col("an.measured_date").alias("analysis_measured_date"),
    )
)

df_enriched.write.mode("overwrite").format("delta").saveAsTable("icare_datapoints_enriched")
print(f"icare_datapoints_enriched: {df_enriched.count()} rows")
df_enriched.show(10, truncate=40)

## 5. Summary stats

In [ ]:
print("DataPoints by measurement type:")
spark.sql("""
    SELECT res_type, global_type, COUNT(*) as count,
           ROUND(AVG(value), 4) as avg_value,
           ROUND(MIN(value), 4) as min_value,
           ROUND(MAX(value), 4) as max_value
    FROM icare_datapoints
    GROUP BY res_type, global_type
    ORDER BY count DESC
    LIMIT 20
""").show(truncate=False)

print("\nAnalyses by technology:")
spark.sql("""
    SELECT technology, current_status, COUNT(*) as count
    FROM icare_analyses
    GROUP BY technology, current_status
    ORDER BY count DESC
    LIMIT 20
""").show(truncate=False)

print("\nTop 10 assets by datapoint count:")
spark.sql("""
    SELECT dp.Asset_id, a.name as asset_name, COUNT(*) as datapoint_count
    FROM icare_datapoints dp
    LEFT JOIN icare_assets a ON dp.Asset_id = a._id
    GROUP BY dp.Asset_id, a.name
    ORDER BY datapoint_count DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
%%sql
select * from icare_datapoints_enriched